# Revio 2.0: Autonomous Multi-Agent Reputation & Crisis Intelligence Engine
### Technical Assessment PoC — AI/ML Engineer Intern for Cynbit Technologies
**Author:** Anurag Pandey  
**Date:** October 2026  

---
## 1. Overview & Setup
This notebook demonstrates the end-to-end AI/ML pipeline for **Revio 2.0**, Cynbit's flagship reputation and sentiment intelligence platform.  
Key capabilities demonstrated:
1. **Aspect-Based Sentiment Analysis (ABSA)** across 6 operational categories with discourse segmentation.
2. **Calibrated Crisis & Urgency Scoring (0-100%)** detecting legal, health/safety, and PR threats.
3. **Multi-Agent Brand Response Generation** with strict legal safety guardrails and Human-in-the-Loop triage.
4. **Multi-Location Analytics** benchmarking regional branches.

In [ ]:
# Import core modules
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import pandas as pd
from src.analyzer import AspectSentimentAnalyzer
from src.crisis_detector import CrisisDetector
from src.response_generator import ResponseGenerator
from src.analytics import EnterpriseAnalyticsEngine
from src.data_generator import load_dataset
from src.config import BrandPersona, CrisisLevel

print("All Revio 2.0 AI Modules successfully loaded!")

## 2. Ingesting Multi-Location Review Benchmark Dataset
We load a benchmark dataset of customer reviews spanning multiple branches (Jaipur, Gurugram, Bengaluru, Mumbai).

In [ ]:
dataset = load_dataset(Path('../data/sample_reviews.json') if Path('../data/sample_reviews.json').exists() else Path('data/sample_reviews.json'))
print(f"Loaded {len(dataset)} customer reviews across {len(set(r['branch_name'] for r in dataset))} branches.")

# Sample review inspection
sample = dataset[2]
print(f"Customer: {sample['customer_name']}")
print(f"Branch:   {sample['branch_name']}")
print(f"Rating:   {sample['star_rating']} Stars")
print(f"Text:     {sample['review_text']}")

## 3. Aspect-Based Sentiment Analysis (ABSA) & Negation Handling
We parse reviews into distinct operational aspects (*Customer Service, Food Quality, Turnaround Speed, Ambience, Pricing, Billing*).

In [ ]:
analyzer = AspectSentimentAnalyzer()

# Test review demonstrating mixed sentiment
test_text = "The truffle pasta was absolutely delicious and fresh! However, the waiting time of 55 minutes was ridiculous and the waiter was rude."
analysis = analyzer.analyze(test_text, star_rating=3)

print(f"Overall Sentiment: {analysis.overall_sentiment.value} (Score: {analysis.compound_score:+.2f})")
print(f"Primary Emotion:   {analysis.primary_emotion}")
print(f"Root-Cause Driver: {analysis.root_cause}")
print("\nExtracted Operational Aspects:")
for a in analysis.detected_aspects:
    print(f"  • Category: {a.category:<25} | Sentiment: {a.sentiment.value:<10} ({a.score:+.2f}) | Excerpt: '{a.context_snippet}'")

## 4. Crisis & Urgency Detection Engine (0-100% Risk Score)
We evaluate an acute crisis review containing severe health and safety allegations.

In [ ]:
detector = CrisisDetector()

crisis_review = (
    "EMERGENCY: Three of us had dinner here last night and woke up with severe food poisoning and vomiting. "
    "My brother is admitted to the hospital right now. The store manager was completely rude and dismissive. "
    "We are reporting this to the municipal food safety authority and filing a police complaint!"
)

crisis_eval = detector.evaluate(crisis_review, star_rating=1)
print(f"Crisis Level:      {crisis_eval.crisis_level.value}")
print(f"Urgency Risk Score:{crisis_eval.risk_score}%")
print(f"Threat Vectors:    {', '.join(crisis_eval.threat_vectors)}")
print(f"Escalation Target: {crisis_eval.escalation_target}")
print(f"SLA Window:        {crisis_eval.sla_window}")
print(f"Action Protocol:   {crisis_eval.recommended_action}")

## 5. Multi-Agent Response Generation with Safety Guardrails
Synthesizes brand-grounded responses and verifies compliance (preventing legal admissions of guilt).

In [ ]:
generator = ResponseGenerator()
crisis_analysis = analyzer.analyze(crisis_review, star_rating=1)

response = generator.generate(
    customer_name="Rohan Malhotra",
    branch_name="Jaipur Malviya Nagar Flagship",
    analysis=crisis_analysis,
    crisis=crisis_eval,
    persona=BrandPersona.EXECUTIVE_ESCALATION
)

print(f"HITL Workflow Status: {response.hitl_status}")
print(f"Triage Reason:        {response.hitl_reason}")
print(f"Tracking Ticket ID:   #{response.ticket_id}")
print(f"Guardrail Passed:     {response.guardrail_passed}")
print("\n--- Generated Response Draft ---")
print(response.response_text)
print("\n--- Actionable Operations Steps ---")
for s in response.actionable_steps:
    print(f" • {s}")

## 6. Enterprise Multi-Branch Analytics
Processes the complete dataset to compute location leaderboards and operational health radar.

In [ ]:
analytics = EnterpriseAnalyticsEngine()
enriched = analytics.process_reviews(dataset)
summary = analytics.generate_executive_summary(enriched)

print("=== Enterprise Network Summary ===")
print(f"Total Reviews Analyzed: {summary['total_reviews']}")
print(f"Network Average Rating: {summary['avg_rating']} Stars")
print(f"Net Sentiment Score:    {summary['avg_sentiment']:+.2f}")
print(f"Crisis Incidents:       {summary['crisis_count']} ({summary['crisis_pct']}%)")

print("\n=== Branch Performance Rankings ===")
rank_df = pd.DataFrame(summary['branch_rankings'])
print(rank_df[['branch_name', 'city', 'avg_rating', 'avg_sentiment', 'crisis_count']].to_string(index=False))